# 🔄 Dataset Augmentation & Class Balancing

## 🎯 Objective

The primary objective of this notebook is to address the severe class imbalance present in the original Indian job listings dataset by integrating an external fake job posting dataset (EMSCAD) and transforming it into an Indian-compatible format.

---

## 📌 Tasks Performed

* Loaded the EMSCAD Fake Job Postings dataset.
* Inspected dataset structure and fraudulent job distribution.
* Extracted fraudulent job postings from the dataset.
* Selected only the relevant features required for this project.
* Mapped EMSCAD columns to match the Indian dataset schema.
* Performed basic Indianization by adapting locations and metadata.
* Generated missing project-specific fields.
* Merged fake job records with the Indian dataset.
* Created a balanced dataset for downstream preprocessing and model training.

---

## 🔍 Key Findings

* The EMSCAD dataset contains a large collection of fraudulent job postings.
* After feature mapping, fake job records become compatible with the Indian dataset structure.
* Dataset balancing significantly improves the availability of fraudulent samples for supervised learning.
* The merged dataset provides a stronger foundation for NLP-based scam detection.

---

## 🚀 Outcome

The notebook produces a unified and balanced dataset containing both legitimate and fraudulent job postings, which will be used in the next phase for data preprocessing and feature engineering.


In [1]:
import pandas as pd

india_df = pd.read_excel("Job_Listings_India_Labelled.xlsx")

fake_df = pd.read_csv("DataSet.csv")

print(india_df.shape)
print(fake_df.shape)

(6553, 13)
(17880, 18)


In [2]:
print(fake_df.head())

                                       title            location department  \
0                           Marketing Intern    US, NY, New York  Marketing   
1  Customer Service - Cloud Video Production      NZ, , Auckland    Success   
2    Commissioning Machinery Assistant (CMA)       US, IA, Wever        NaN   
3          Account Executive - Washington DC  US, DC, Washington      Sales   
4                        Bill Review Manager  US, FL, Fort Worth        NaN   

  salary_range                                    company_profile  \
0          NaN  <h3>We're Food52, and we've created a groundbr...   
1          NaN  <h3>90 Seconds, the worlds Cloud Video Product...   
2          NaN  <h3></h3>\r\n<p>Valor Services provides Workfo...   
3          NaN  <p>Our passion for improving quality of life t...   
4          NaN  <p>SpotSource Solutions LLC is a Global Human ...   

                                         description  \
0  <p>Food52, a fast-growing, James Beard Award-w...  

In [3]:
print("\nINDIA DATASET COLUMNS\n")
print(india_df.columns.tolist())

print("\n\nEMSCAD DATASET COLUMNS\n")
print(fake_df.columns.tolist())


INDIA DATASET COLUMNS

['City', 'Listing Type', 'Field', 'Job Title', 'Company Name', 'Salary/Stipend', 'Salary_Amount', 'Job Description', 'Location', 'Posted Date', 'Experience Level', 'Application Link', 'Real/Fake Job']


EMSCAD DATASET COLUMNS

['title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent', 'in_balanced_dataset']


Checking Target Distribution of EMSCAD Dataset

In [4]:
print(fake_df["fraudulent"].value_counts())

print()

print(fake_df["fraudulent"].value_counts(normalize=True)*100)

fraudulent
f    17014
t      866
Name: count, dtype: int64

fraudulent
f    95.1566
t     4.8434
Name: proportion, dtype: float64


Chekcing Missing values

In [5]:
missing = pd.DataFrame({
    "Missing":fake_df.isnull().sum(),
    "Percent":round(fake_df.isnull().sum()/len(fake_df)*100,2)
})

missing.sort_values("Percent",ascending=False)

,Missing,Percent
salary_range,15012,83.96
department,11547,64.58
required_education,8105,45.33
benefits,7196,40.25
required_experience,7050,39.43
function,6455,36.10
industry,4903,27.42
employment_type,3471,19.41
company_profile,3308,18.50
requirements,2689,15.04


Duplicates

In [6]:
print(fake_df.duplicated().sum())

235


In [7]:
fake_df[[
"title",
"description",
"requirements",
"salary_range",
"employment_type",
"required_experience",
"industry",
"fraudulent"
]].head(10)

,title,description,requirements,salary_range,employment_type,required_experience,industry,fraudulent
0,Marketing Intern,"<p>Food52, a fast-growing, James Beard Award-w...",<ul>\r\n<li>Experience with content management...,NaN,Other,Internship,NaN,f
1,Customer Service - Cloud Video Production,<p>Organised - Focused - Vibrant - Awesome!<br...,<p><b>What we expect from you:</b></p>\r\n<p>Y...,NaN,Full-time,Not Applicable,Marketing and Advertising,f
2,Commissioning Machinery Assistant (CMA),"<p>Our client, located in Houston, is actively...",<ul>\r\n<li>Implement pre-commissioning and co...,NaN,NaN,NaN,NaN,f
3,Account Executive - Washington DC,<p><b>THE COMPANY: ESRI – Environmental System...,<ul>\r\n<li>\r\n<b>EDUCATION: </b>Bachelor’s o...,NaN,Full-time,Mid-Senior level,Computer Software,f
4,Bill Review Manager,<p><b>JOB TITLE:</b> Itemization Review Manage...,<p><b>QUALIFICATIONS:</b></p>\r\n<ul>\r\n<li>R...,NaN,Full-time,Mid-Senior level,Hospital & Health Care,f
5,Accounting Clerk,<p><b>Job Overview</b></p>\r\n<p>Apex is an en...,NaN,NaN,NaN,NaN,NaN,f
6,Head of Content (m/f),<p><b>Your Responsibilities:</b></p>\r\n<p> </...,<p><b>Your Know-How:</b></p>\r\n<p><b> ...,20000-28000,Full-time,Mid-Senior level,Online Media,f
7,Lead Guest Service Specialist,<h3>Who is Airenvy?</h3>\r\n<p>Hey there! We a...,"<ul>\r\n<li>Experience with CRM software, live...",NaN,NaN,NaN,NaN,f
8,HP BSM SME,<p></p>\r\n<p></p>\r\n<p>Implementation/Config...,<p><b>MUST BE A US CITIZEN.</b></p>\r\n<p><b>A...,NaN,Full-time,Associate,Information Technology and Services,f
9,Customer Service Associate - Part Time,<p>The Customer Service Associate will be base...,<p><b>Minimum Requirements:</b></p>\r\n<ul>\r\...,NaN,Part-time,Entry level,Financial Services,f


In [8]:
fake_jobs = fake_df[fake_df["fraudulent"]=="t"]

print(fake_jobs.shape)

(866, 18)


Only usefull columns because this dataset contains US jobs 

In [9]:
fake_jobs = fake_jobs[
[
"title",
"description",
"requirements",
"employment_type",
"required_experience",
"industry",
"location"
]
]

Phase-4: Indianization + Column Mapping

In [10]:
location_mapping = {
    "New York": "Delhi",
    "California": "Bangalore",
    "Texas": "Hyderabad",
    "Washington": "Mumbai",
    "Florida": "Pune",
    "Chicago": "Chennai",
    "Los Angeles": "Bangalore",
    "San Francisco": "Bangalore",
    "Seattle": "Hyderabad",
    "Boston": "Gurugram"
}

for us_city, india_city in location_mapping.items():
    fake_jobs["location"] = fake_jobs["location"].str.replace(
        us_city, india_city, case=False, regex=False
    )

fake_jobs["location"] = fake_jobs["location"].fillna("Remote")

Descritpion Indianize

In [11]:
replacements = {
    "USA": "India",
    "US": "India",
    "United States": "India",
    "Dollar": "Rupees",
    "USD": "INR",
    "$": "₹"
}

for old, new in replacements.items():
    fake_jobs["description"] = fake_jobs["description"].str.replace(
        old, new, case=False, regex=False
    )

    fake_jobs["requirements"] = fake_jobs["requirements"].str.replace(
        old, new, case=False, regex=False
    )

Combine Description + Requirements

In [12]:
fake_jobs["Job Description"] = (
    fake_jobs["description"].fillna("") +
    " " +
    fake_jobs["requirements"].fillna("")
)

Rename Columns

In [13]:
fake_jobs = fake_jobs.rename(columns={
    "title": "Job Title",
    "industry": "Field",
    "employment_type": "Listing Type",
    "required_experience": "Experience Level",
    "location": "Location"
})

Create Missing Columns

In [14]:
import random

cities = [
    "Delhi","Mumbai","Bangalore","Hyderabad",
    "Pune","Chennai","Kolkata","Noida","Gurugram"
]

fake_jobs["City"] = [random.choice(cities) for _ in range(len(fake_jobs))]

fake_jobs["Company Name"] = "Unknown Company"

fake_jobs["Salary/Stipend"] = "Not specified"

fake_jobs["Salary_Amount"] = None

fake_jobs["Posted Date"] = "2025-01-01"

fake_jobs["Application Link"] = "https://fakejobportal.in"

fake_jobs["Real/Fake Job"] = "Fake Job"


Arranging columns like Indian Dataset

In [15]:
fake_jobs = fake_jobs[
[
'City',
'Listing Type',
'Field',
'Job Title',
'Company Name',
'Salary/Stipend',
'Salary_Amount',
'Job Description',
'Location',
'Posted Date',
'Experience Level',
'Application Link',
'Real/Fake Job'
]
]

Chekcing Finalized Dataset

In [16]:
print(fake_jobs.head())

print(fake_jobs.shape)

print(fake_jobs.columns)

         City Listing Type                                Field  \
98    Chennai    Full-time                         Oil & Energy   
144  Gurugram          NaN                                  NaN   
173   Kolkata    Full-time  Electrical/Electronic Manufacturing   
180   Chennai          NaN                                  NaN   
215      Pune    Full-time                         Oil & Energy   

                             Job Title     Company Name Salary/Stipend  \
98                     IC&E Technician  Unknown Company  Not specified   
144                       Forward Cap.  Unknown Company  Not specified   
173  Technician Instrument & Controls   Unknown Company  Not specified   
180                    Sales Executive  Unknown Company  Not specified   
215            IC&E Technician Mt Poso  Unknown Company  Not specified   

    Salary_Amount                                    Job Description  \
98           None  <p><b><img src="#URL_ae07dc35dfe86ebc1101b48ee...   
144     

In [17]:
print(fake_jobs.columns.tolist())

['City', 'Listing Type', 'Field', 'Job Title', 'Company Name', 'Salary/Stipend', 'Salary_Amount', 'Job Description', 'Location', 'Posted Date', 'Experience Level', 'Application Link', 'Real/Fake Job']


Location change to Indian based

In [18]:
import re

# US aur country codes replace
fake_jobs["Location"] = fake_jobs["Location"].fillna("Remote")

fake_jobs["Location"] = fake_jobs["Location"].str.replace("US", "India", regex=False)
fake_jobs["Location"] = fake_jobs["Location"].str.replace("CA", "Delhi", regex=False)
fake_jobs["Location"] = fake_jobs["Location"].str.replace("NY", "Mumbai", regex=False)
fake_jobs["Location"] = fake_jobs["Location"].str.replace("TX", "Hyderabad", regex=False)
fake_jobs["Location"] = fake_jobs["Location"].str.replace("FL", "Pune", regex=False)

# Extra commas remove
fake_jobs["Location"] = fake_jobs["Location"].str.replace(r"\s*,\s*", ", ", regex=True)

Filling Missing Feild

In [19]:
fake_jobs["Field"] = fake_jobs["Field"].fillna("Information Technology")

Missing Listing type

In [20]:
fake_jobs["Listing Type"] = fake_jobs["Listing Type"].fillna("Full-time")

Missing Experience

In [21]:
fake_jobs["Experience Level"] = fake_jobs["Experience Level"].fillna("Entry Level (0-2 years)")

Making company name realistic

In [22]:
import random

companies = [
    "TechNova Solutions",
    "FutureSoft Pvt Ltd",
    "DigitalEdge Technologies",
    "NextGen Systems",
    "Skyline IT Services",
    "Innovix Solutions",
    "CodeCraft India",
    "VisionTech Pvt Ltd",
    "CloudNova Solutions",
    "Apex Digital Services"
]

fake_jobs["Company Name"] = [
    random.choice(companies)
    for _ in range(len(fake_jobs))
]

Merging Both Datasets

In [23]:
balanced_df = pd.concat(
    [india_df, fake_jobs],
    ignore_index=True
)

balanced_df = balanced_df.drop_duplicates()

print(balanced_df.shape)

print(balanced_df["Real/Fake Job"].value_counts())

(7419, 13)
Real/Fake Job
Real Job    6532
Fake Job     887
Name: count, dtype: int64


C:\Users\Raghvendra Goyal\AppData\Local\Temp\ipykernel_25184\2897724355.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  balanced_df = pd.concat(


Saving the new Balanced Dataset

In [24]:
balanced_df.to_csv(
    "balanced_dataset.csv",
    index=False
)

print("Dataset Saved done")

Dataset Saved done
